# Example 10.4.3 / 5: Rewrite Exactly through Data

Companion notebook for **10.4.3 Automatic Piecewise-Linear Approximation** in the AMPL lab Documentation, section "Rewrite Exactly through Data". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Rewrite Exactly through Data

Prefer an exact formulation when one exists. In the `Growth` model the function is monotone and
the right-hand side is data, so `exp(Growth) <= budget` is the same as `Growth <= log(budget)`,
and AMPL evaluates `log(budget)` before the solve. Nothing is approximated:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param budget > 0;  # upper limit on exp(Growth)
var Growth;  # no bounds, and none needed

maximize Level: Growth;  # push Growth as high as the budget allows
subject to BudgetLimit: Growth <= log(budget);  # same limit, linear in Growth

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["budget"] = 1e8

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print(ampl.var["Growth"].value())

solved
18.42068074395237


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param budget := 1e8;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display Growth;

Growth = 18.4207

